# 🎵 Raga Bhairav: Unconditioned Melody Generation with LSTM

**Task 1 — Symbolic, Unconditioned Generation**

We train an LSTM to learn the melodic distribution of **Raga Bhairav**, one of the oldest and most revered ragas in Indian classical music. The model learns from synthetically generated sequences that encode Bhairav's grammar — its ascent/descent scale, characteristic phrases, and note hierarchy — and then generates entirely new melodies by sampling from the learned distribution.

### What is Raga Bhairav?
- **Time**: Performed at dawn
- **Mood (rasa)**: Serious, devotional, profound
- **Key feature**: Uses both flat 2nd (komal Re) and flat 6th (komal Dha) — giving it a unique, ancient sound
- **Arohana** (ascent):  S r G m P d N S'
- **Avarohana** (descent): S' N d P m G r S
- **Vadi** (most important note): M (Ma)
- **Samvadi** (second most important): S (Sa)

### Pipeline Overview
1. Encode Bhairav grammar → generate synthetic training sequences
2. Tokenize sequences (pitch + duration)
3. Train a 2-layer LSTM with embedding
4. Sample new melodies via temperature-controlled generation
5. Export to MIDI → convert to MP3

## 0. Install Dependencies

In [ ]:
# Run this cell first on Google Colab
!pip install music21 -q
!apt-get install -y musescore3 fluidsynth fluid-soundfont-gm -q
!pip install pyfluidsynth -q

## 1. Imports & Setup

In [ ]:
import random
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from music21 import stream, note, tempo, instrument, midi
from collections import Counter
import os
import subprocess

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {DEVICE}")

## 2. Raga Bhairav Grammar

We encode Bhairav's rules explicitly:
- **Scale**: Sa, komal Re, Ga, Ma, Pa, komal Dha, Ni (in MIDI: C, Db, E, F, G, Ab, B)
- **Arohana/Avarohana**: Separate ascending and descending note orderings
- **Vadi/Samvadi**: Ma and Sa get higher selection probability
- **Characteristic phrases (pakad)**: Seed sequences that define the raga's identity
- **Gamaka**: Ornamental note patterns common in Indian classical music

In [ ]:
# ── Raga Bhairav in MIDI pitch numbers (octave 4, middle octave) ──
# Sa=C4, komal Re=Db4, Ga=E4, Ma=F4, Pa=G4, komal Dha=Ab4, Ni=B4

BHAIRAV = {
    'name': 'Bhairav',
    # Arohana: ascending scale
    'arohana': [60, 61, 64, 65, 67, 68, 71, 72],  # C Db E F G Ab B C'
    # Avarohana: descending scale
    'avarohana': [72, 71, 68, 67, 65, 64, 61, 60],  # C' B Ab G F E Db C
    # Full note set across 2 octaves for generation
    'notes': [48, 49, 52, 53, 55, 56, 59,   # octave 3
              60, 61, 64, 65, 67, 68, 71,   # octave 4
              72, 73, 76, 77, 79, 80, 83],  # octave 5
    # Vadi (most important) and Samvadi (second most important)
    'vadi': [65, 53, 77],      # Ma (F) across octaves
    'samvadi': [60, 48, 72],   # Sa (C) across octaves
    # Characteristic pakad phrases (note sequences that define Bhairav)
    'pakad': [
        [61, 60, 68, 67, 65],           # Re Sa Dha Pa Ma — classic opening
        [64, 65, 67, 68, 67, 65],       # Ga Ma Pa Dha Pa Ma
        [71, 72, 71, 68, 67],           # Ni Sa' Ni Dha Pa — upper movement
        [60, 61, 64, 65, 64, 61, 60],   # Sa Re Ga Ma Ga Re Sa — full phrase
        [65, 67, 68, 71, 72],           # Ma Pa Dha Ni Sa' — ascending
        [72, 71, 68, 67, 65, 64, 61, 60], # full avarohana
    ],
    # Gamaka: ornamental turns around key notes
    'gamaka': [
        [65, 64, 65],    # Ma Ga Ma (oscillation)
        [67, 68, 67],    # Pa Dha Pa
        [61, 60, 61],    # Re Sa Re
        [71, 72, 71],    # Ni Sa' Ni
    ]
}

# Duration vocabulary (in quarter note units)
DURATIONS = [0.25, 0.5, 0.75, 1.0, 1.5, 2.0]
DUR_WEIGHTS = [0.15, 0.35, 0.10, 0.25, 0.10, 0.05]  # 0.5 and 1.0 most common

print(f"Raga: {BHAIRAV['name']}")
print(f"Scale size: {len(set(BHAIRAV['notes']))} unique pitches across 3 octaves")
print(f"Pakad phrases: {len(BHAIRAV['pakad'])}")
print(f"Duration vocabulary: {DURATIONS}")

## 3. Synthetic Data Generation

We generate training sequences using a **grammar-constrained random walk**:
- Each sequence starts from Sa or a pakad phrase
- Notes are sampled from the raga scale with vadi/samvadi bias
- Direction (ascending/descending) shifts probabilistically
- Pakad phrases and gamakas are injected randomly to add authentic character
- Each token is a `(pitch, duration)` pair

In [ ]:
def get_note_weights(current_pitch, raga, direction):
    """Return sampling weights for each note in the raga scale.
    Biases towards: stepwise motion, vadi/samvadi notes, current direction."""
    scale = raga['notes']
    weights = []
    for p in scale:
        w = 1.0
        interval = p - current_pitch
        # Prefer stepwise motion (small intervals)
        if abs(interval) <= 2:
            w *= 3.0
        elif abs(interval) <= 4:
            w *= 1.5
        else:
            w *= 0.4
        # Prefer notes in current direction
        if direction == 'up' and interval > 0:
            w *= 1.8
        elif direction == 'down' and interval < 0:
            w *= 1.8
        # Boost vadi and samvadi
        if p in raga['vadi']:
            w *= 2.5
        if p in raga['samvadi']:
            w *= 1.8
        # Avoid staying on same note too long
        if p == current_pitch:
            w *= 0.3
        weights.append(w)
    return weights


def generate_sequence(raga, length=64):
    """Generate a single melody sequence as list of (pitch, duration) tokens."""
    sequence = []
    scale = raga['notes']

    # Start on Sa (tonic) or begin with a pakad phrase
    if random.random() < 0.4:
        pakad = random.choice(raga['pakad'])
        for p in pakad:
            d = random.choices(DURATIONS, weights=DUR_WEIGHTS)[0]
            sequence.append((p, d))
        current = pakad[-1]
    else:
        current = random.choice(raga['samvadi'])  # start on Sa
        d = random.choices(DURATIONS, weights=DUR_WEIGHTS)[0]
        sequence.append((current, d))

    direction = random.choice(['up', 'down'])
    dir_steps = 0  # steps taken in current direction

    while len(sequence) < length:
        # Randomly inject a gamaka ornament
        if random.random() < 0.08:
            gamaka = random.choice(raga['gamaka'])
            for p in gamaka:
                if p in scale:
                    sequence.append((p, 0.25))  # gamakas are fast
            current = gamaka[-1]
            continue

        # Randomly inject a pakad phrase
        if random.random() < 0.06:
            pakad = random.choice(raga['pakad'])
            for p in pakad:
                d = random.choices(DURATIONS, weights=DUR_WEIGHTS)[0]
                sequence.append((p, d))
            current = pakad[-1]
            continue

        # Possibly flip direction
        dir_steps += 1
        if dir_steps > random.randint(4, 10):
            direction = 'down' if direction == 'up' else 'up'
            dir_steps = 0

        # Sample next note
        weights = get_note_weights(current, raga, direction)
        total = sum(weights)
        probs = [w / total for w in weights]
        next_pitch = random.choices(scale, weights=probs)[0]
        next_dur = random.choices(DURATIONS, weights=DUR_WEIGHTS)[0]

        sequence.append((next_pitch, next_dur))
        current = next_pitch

    return sequence[:length]


# Generate training data
NUM_SEQUENCES = 2000
SEQ_LENGTH = 64

print(f"Generating {NUM_SEQUENCES} sequences of length {SEQ_LENGTH}...")
all_sequences = [generate_sequence(BHAIRAV, SEQ_LENGTH) for _ in range(NUM_SEQUENCES)]
print(f"Done. Total tokens: {NUM_SEQUENCES * SEQ_LENGTH:,}")

# Show a sample
print("\nSample sequence (first 10 tokens):")
for pitch, dur in all_sequences[0][:10]:
    print(f"  MIDI {pitch} ({note.Note(pitch).nameWithOctave}), duration={dur}")

## 4. Tokenization

Each `(pitch, duration)` pair becomes a single integer token. This gives us a small, manageable vocabulary that the LSTM can learn efficiently.

In [ ]:
# Build vocabulary: every unique (pitch, duration) pair
all_tokens = [token for seq in all_sequences for token in seq]
vocab = sorted(set(all_tokens))
token2idx = {t: i for i, t in enumerate(vocab)}
idx2token = {i: t for t, i in token2idx.items()}
VOCAB_SIZE = len(vocab)

print(f"Vocabulary size: {VOCAB_SIZE} unique (pitch, duration) pairs")
print(f"Example tokens: {vocab[:5]}")

# Convert all sequences to integer indices
indexed_sequences = [[token2idx[t] for t in seq] for seq in all_sequences]

# Token frequency analysis
freq = Counter(all_tokens)
print(f"\nTop 5 most common tokens:")
for tok, count in freq.most_common(5):
    p, d = tok
    print(f"  {note.Note(p).nameWithOctave}, dur={d} → {count} times ({100*count/len(all_tokens):.1f}%)")

## 5. PyTorch Dataset

We use a **sliding window** approach: each training sample is a window of `WINDOW_SIZE` tokens as input, and the next token as the target. This is standard next-token prediction (same as language modeling).

In [ ]:
WINDOW_SIZE = 32  # context length

class RagaDataset(Dataset):
    def __init__(self, sequences, window_size):
        self.samples = []
        for seq in sequences:
            for i in range(len(seq) - window_size):
                x = seq[i : i + window_size]
                y = seq[i + window_size]
                self.samples.append((x, y))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        x, y = self.samples[idx]
        return torch.tensor(x, dtype=torch.long), torch.tensor(y, dtype=torch.long)


# Train/val split (90/10)
split = int(0.9 * len(indexed_sequences))
train_seqs = indexed_sequences[:split]
val_seqs   = indexed_sequences[split:]

train_dataset = RagaDataset(train_seqs, WINDOW_SIZE)
val_dataset   = RagaDataset(val_seqs,   WINDOW_SIZE)

train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True,  num_workers=2)
val_loader   = DataLoader(val_dataset,   batch_size=128, shuffle=False, num_workers=2)

print(f"Training samples:   {len(train_dataset):,}")
print(f"Validation samples: {len(val_dataset):,}")
print(f"Batches per epoch:  {len(train_loader):,}")

## 6. LSTM Model

Architecture:
- **Embedding layer**: maps each token index to a 64-dim vector
- **2-layer LSTM**: 256 hidden units, dropout between layers for regularization
- **Linear output head**: projects to vocabulary size → softmax probabilities

This is intentionally small — it trains fast on Colab and the vocabulary is compact enough that it doesn't need to be large.

In [ ]:
class RagaLSTM(nn.Module):
    def __init__(self, vocab_size, embed_dim=64, hidden_dim=256, num_layers=2, dropout=0.3):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.lstm = nn.LSTM(
            input_size=embed_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            dropout=dropout,
            batch_first=True
        )
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_dim, vocab_size)

    def forward(self, x, hidden=None):
        # x: (batch, seq_len)
        emb = self.dropout(self.embedding(x))     # (batch, seq_len, embed_dim)
        out, hidden = self.lstm(emb, hidden)      # (batch, seq_len, hidden_dim)
        out = self.dropout(out[:, -1, :])         # take last timestep
        logits = self.fc(out)                     # (batch, vocab_size)
        return logits, hidden


model = RagaLSTM(vocab_size=VOCAB_SIZE).to(DEVICE)

total_params = sum(p.numel() for p in model.parameters())
print(f"Model parameters: {total_params:,}")
print(model)

## 7. Training

Standard next-token cross-entropy loss with Adam optimizer. We use a learning rate scheduler to decay LR when validation loss plateaus.

In [ ]:
EPOCHS = 30
LR = 1e-3

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LR)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=3, factor=0.5)

train_losses, val_losses = [], []
best_val_loss = float('inf')

for epoch in range(1, EPOCHS + 1):
    # ── Training ──
    model.train()
    total_loss = 0
    for x_batch, y_batch in train_loader:
        x_batch, y_batch = x_batch.to(DEVICE), y_batch.to(DEVICE)
        optimizer.zero_grad()
        logits, _ = model(x_batch)
        loss = criterion(logits, y_batch)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)  # gradient clipping
        optimizer.step()
        total_loss += loss.item()
    avg_train = total_loss / len(train_loader)

    # ── Validation ──
    model.eval()
    total_val = 0
    with torch.no_grad():
        for x_batch, y_batch in val_loader:
            x_batch, y_batch = x_batch.to(DEVICE), y_batch.to(DEVICE)
            logits, _ = model(x_batch)
            total_val += criterion(logits, y_batch).item()
    avg_val = total_val / len(val_loader)

    scheduler.step(avg_val)
    train_losses.append(avg_train)
    val_losses.append(avg_val)

    if avg_val < best_val_loss:
        best_val_loss = avg_val
        torch.save(model.state_dict(), 'best_model.pt')

    if epoch % 5 == 0 or epoch == 1:
        print(f"Epoch {epoch:02d}/{EPOCHS} | Train Loss: {avg_train:.4f} | Val Loss: {avg_val:.4f} | LR: {optimizer.param_groups[0]['lr']:.6f}")

print(f"\nBest validation loss: {best_val_loss:.4f}")

## 8. Training Curves

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 4))
plt.plot(train_losses, label='Train Loss', color='steelblue')
plt.plot(val_losses,   label='Val Loss',   color='coral')
plt.xlabel('Epoch')
plt.ylabel('Cross-Entropy Loss')
plt.title('Raga Bhairav LSTM — Training Curves')
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('training_curves.png', dpi=150)
plt.show()
print("Saved training_curves.png")

## 9. Melody Generation

We sample from the model using **temperature scaling**:
- **Low temperature (0.5)**: conservative, stays close to learned grammar
- **Medium temperature (1.0)**: balanced creativity
- **High temperature (1.5)**: more adventurous, occasionally breaks rules

Generation starts from a seed pakad phrase to ground the melody in Bhairav's identity.

In [ ]:
def generate_melody(model, seed_tokens, length=128, temperature=1.0):
    """Autoregressively generate a melody from a seed sequence."""
    model.eval()
    generated = list(seed_tokens)

    with torch.no_grad():
        for _ in range(length):
            # Use last WINDOW_SIZE tokens as context
            context = generated[-WINDOW_SIZE:]
            x = torch.tensor([context], dtype=torch.long).to(DEVICE)
            logits, _ = model(x)

            # Temperature scaling
            logits = logits[0] / temperature
            probs = torch.softmax(logits, dim=-1)

            # Sample next token
            next_idx = torch.multinomial(probs, 1).item()
            generated.append(next_idx)

    # Convert back to (pitch, duration) tokens, skip seed
    return [idx2token[i] for i in generated[len(seed_tokens):]]


# Load best checkpoint
model.load_state_dict(torch.load('best_model.pt', map_location=DEVICE))

# Seed with the classic Bhairav opening pakad
seed_phrase = BHAIRAV['pakad'][0]  # [Re Sa Dha Pa Ma]
seed_tokens = [token2idx[(p, 0.5)] if (p, 0.5) in token2idx
               else token2idx[min(token2idx.keys(), key=lambda t: abs(t[0]-p))]
               for p in seed_phrase]

# Generate at three temperatures
melodies = {}
for temp in [0.5, 1.0, 1.5]:
    mel = generate_melody(model, seed_tokens, length=96, temperature=temp)
    melodies[temp] = mel
    print(f"Temperature {temp}: generated {len(mel)} tokens")
    print(f"  First 5: {[(note.Note(p).nameWithOctave, d) for p,d in mel[:5]]}")

## 10. Export to MIDI

In [ ]:
def melody_to_midi(tokens, filename, bpm=60, instrument_name='Sitar'):
    """Convert (pitch, duration) token list to a MIDI file."""
    s = stream.Score()
    part = stream.Part()

    # Set instrument and tempo
    part.append(instrument.Sitar())  # Sitar for authentic Indian feel
    part.append(tempo.MetronomeMark(number=bpm))

    for pitch_midi, duration in tokens:
        n = note.Note(pitch_midi)
        n.quarterLength = duration
        part.append(n)

    s.append(part)
    s.write('midi', fp=filename)
    print(f"Saved MIDI: {filename}")
    return filename


# Export all three temperatures
for temp, mel in melodies.items():
    melody_to_midi(mel, f'bhairav_temp{temp}.mid', bpm=55)

## 11. Convert MIDI → MP3

We use **FluidSynth** with a General MIDI soundfont to render the MIDI to audio, then convert to MP3 with ffmpeg.

In [ ]:
SOUNDFONT = '/usr/share/sounds/sf2/FluidR3_GM.sf2'  # installed via apt above

def midi_to_mp3(midi_path, mp3_path, soundfont=SOUNDFONT):
    """Convert MIDI to MP3 via FluidSynth + ffmpeg."""
    wav_path = midi_path.replace('.mid', '.wav')
    # Render to WAV with FluidSynth
    subprocess.run([
        'fluidsynth', '-ni', soundfont, midi_path,
        '-F', wav_path, '-r', '44100'
    ], check=True, capture_output=True)
    # Convert WAV to MP3 with ffmpeg
    subprocess.run([
        'ffmpeg', '-y', '-i', wav_path,
        '-codec:a', 'libmp3lame', '-qscale:a', '2',
        mp3_path
    ], check=True, capture_output=True)
    os.remove(wav_path)
    print(f"Saved MP3: {mp3_path}")


for temp in [0.5, 1.0, 1.5]:
    midi_to_mp3(f'bhairav_temp{temp}.mid', f'bhairav_temp{temp}.mp3')

print("\nAll MP3s generated!")

## 12. Listen in Notebook

In [ ]:
from IPython.display import Audio, display

for temp in [0.5, 1.0, 1.5]:
    print(f"\n🎵 Temperature = {temp}")
    display(Audio(f'bhairav_temp{temp}.mp3'))

## 13. Analysis: Did the Model Learn Bhairav?

We verify the model has internalized Bhairav's grammar by checking:
1. **Scale adherence**: what % of generated notes are in the Bhairav scale?
2. **Vadi emphasis**: is Ma (F) the most frequent note?
3. **Interval distribution**: does the model prefer stepwise motion?

In [ ]:
bhairav_pitch_classes = {p % 12 for p in BHAIRAV['notes']}
# C=0, Db=1, E=4, F=5, G=7, Ab=8, B=11
note_names = {0:'Sa(C)', 1:'Re(Db)', 4:'Ga(E)', 5:'Ma(F)',
              7:'Pa(G)', 8:'Dha(Ab)', 11:'Ni(B)'}

print("=" * 50)
print("GENERATED MELODY ANALYSIS (Temperature = 1.0)")
print("=" * 50)

mel = melodies[1.0]
pitches = [p for p, d in mel]
pitch_classes = [p % 12 for p in pitches]

# Scale adherence
in_scale = sum(1 for pc in pitch_classes if pc in bhairav_pitch_classes)
print(f"\nScale adherence: {in_scale}/{len(pitches)} = {100*in_scale/len(pitches):.1f}%")

# Note frequency
pc_freq = Counter(pitch_classes)
print("\nNote frequency (pitch class):")
for pc, count in pc_freq.most_common():
    name = note_names.get(pc, f'pc={pc}')
    bar = '█' * int(30 * count / len(pitches))
    print(f"  {name:12s} {bar} {count}")

# Interval distribution
intervals = [abs(pitches[i+1] - pitches[i]) for i in range(len(pitches)-1)]
interval_freq = Counter(intervals)
print("\nInterval distribution (semitones):")
for interval in sorted(interval_freq.keys())[:8]:
    count = interval_freq[interval]
    bar = '█' * int(30 * count / len(intervals))
    print(f"  {interval:2d} semitones  {bar} {100*count/len(intervals):.1f}%")

## 14. Save Everything for Submission

In [ ]:
# Save model checkpoint
torch.save({
    'model_state_dict': model.state_dict(),
    'vocab': vocab,
    'token2idx': token2idx,
    'idx2token': idx2token,
    'window_size': WINDOW_SIZE,
    'raga': BHAIRAV['name'],
}, 'raga_bhairav_checkpoint.pt')

print("Saved files:")
for f in ['raga_bhairav_checkpoint.pt',
          'bhairav_temp0.5.mid', 'bhairav_temp1.0.mid', 'bhairav_temp1.5.mid',
          'bhairav_temp0.5.mp3', 'bhairav_temp1.0.mp3', 'bhairav_temp1.5.mp3',
          'training_curves.png']:
    if os.path.exists(f):
        size = os.path.getsize(f)
        print(f"  ✅ {f} ({size:,} bytes)")
    else:
        print(f"  ❌ {f} (not found)")